# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

## Baseline Rule

Prioritize pages that already receive search impressions but have low CTR, are outdated, or have declining traffic.

### Reason Codes

- LOW_CTR: CTR is below 3%.
- HIGH_IMPRESSIONS: More than 1000 impressions in the last 90 days.
- STALE_CONTENT: Last updated more than 180 days ago.
- FALLING_TREND: Traffic trend is down.
- GOOD_POSITION: Average Google position is below 20.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [10]:
import pandas as pd
from pathlib import Path

DATA_PATH = Path("../../data/raw/content_refresh_anonymized.csv")
df = pd.read_csv(DATA_PATH)


def score_row(row):
    score = 0
    reasons = []

    # Low CTR
    if row["ctr"] < 3:
        score += 2
        reasons.append("LOW_CTR")

    # High impressions
    if row["impressions_90d"] > 1000:
        score += 2
        reasons.append("HIGH_IMPRESSIONS")

    # Old content
    if row["days_since_last_update"] > 180:
        score += 2
        reasons.append("STALE_CONTENT")

    # Falling traffic
    if row["trend_direction"] == "down":
        score += 1
        reasons.append("FALLING_TREND")

    # Already ranking reasonably well
    if row["avg_position"] < 20:
        score += 1
        reasons.append("GOOD_POSITION")

    # Decide action
    if score >= 6:
        action = "High Priority"
    elif score >= 3:
        action = "Medium Priority"
    else:
        action = "Low Priority"

    return pd.Series({
        "score": score,
        "reason_code": ", ".join(reasons),
        "action": action
    })


df[["score", "reason_code", "action"]] = df.apply(score_row, axis=1)

df = df.sort_values(
    by=["score", "impressions_90d"],
    ascending=False
)

OUTPUT = Path("../outputs/baseline_action_score.csv")
OUTPUT.parent.mkdir(parents=True, exist_ok=True)

df.to_csv(OUTPUT, index=False)

print("Saved to:", OUTPUT)

df.head(20)

Saved to: ..\outputs\baseline_action_score.csv


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct,score,reason_code,action
16751,content_cf56e2e2e282,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,5125.0,33705.0,...,0.84,24.11,0.00,excellent,striking,down,-85.6,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
21268,content_0a91db491d14,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3478.0,21948.0,...,5.13,41.76,0.00,good,striking,down,-51.8,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
12045,content_c2d929d83eaa,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4758.0,30070.0,...,0.00,40.00,0.00,good,striking,down,-62.8,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
5327,content_fe16a55cd13d,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3388.0,21742.0,...,2.38,38.64,0.00,good,striking,down,-52.2,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
20837,content_928af3e22c80,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3118.0,20396.0,...,0.00,0.00,0.00,moderate,striking,down,-45.7,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
22872,content_e3ff1b093148,client_d029fa3a95,0.0,0.00,LOW,0.00,keyword article,informational,4758.0,33575.0,...,0.00,20.00,0.00,moderate,page_1,down,-68.5,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
16514,content_7368877ea310,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,2591.0,16498.0,...,3.66,42.99,0.00,excellent,page_3_5,down,-81.5,7,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
7021,content_1bfaa38ff26c,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3861.0,24672.0,...,3.75,43.33,0.00,good,page_3_5,down,-74.7,7,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
11489,content_5feee3994adb,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,transactional,3590.0,22780.0,...,0.00,40.00,0.00,good,page_3_5,down,-89.1,7,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority
698,content_b16bd7307b39,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,4329.0,27844.0,...,0.00,25.00,0.00,good,page_3_5,down,-69.7,7,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",High Priority


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [11]:
top20 = df.head(20).copy()

top20["confidence"] = "Medium"

top20["what_would_make_it_wrong"] = (
    "Traffic may be seasonal or influenced by external events."
)

top20[
    [
        "content_id",
        "action",
        "score",
        "reason_code",
        "confidence",
        "what_would_make_it_wrong"
    ]
]

,content_id,action,score,reason_code,confidence,what_would_make_it_wrong
16751,content_cf56e2e2e282,High Priority,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
21268,content_0a91db491d14,High Priority,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
12045,content_c2d929d83eaa,High Priority,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
5327,content_fe16a55cd13d,High Priority,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
20837,content_928af3e22c80,High Priority,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
22872,content_e3ff1b093148,High Priority,8,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
16514,content_7368877ea310,High Priority,7,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
7021,content_1bfaa38ff26c,High Priority,7,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
11489,content_5feee3994adb,High Priority,7,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...
698,content_b16bd7307b39,High Priority,7,"LOW_CTR, HIGH_IMPRESSIONS, STALE_CONTENT, FALL...",Medium,Traffic may be seasonal or influenced by exter...


## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

## Weak Picks

Some pages may receive high scores because they have low CTR but limited search demand.
Some traffic declines may be caused by seasonality rather than content quality.

## Leakage Check

- Only historical metrics were used.
- No future information was used.
- No label-derived features were included.
- No product-specific flags were used.
- The rule relies only on observable data available at scoring time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.